# CLM experiments
Qwen2.5-0.5B embeddings → small contrastive heads on Natural Questions.
Queries are states, answers are actions.

## Setup

In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
from datasets import load_dataset
import torch.nn as nn


c:\Users\afeef\OneDrive\Desktop\My Folder\Codes\ML\clm-experiments\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
model = AutoModel.from_pretrained("Qwen/Qwen2.5-0.5B", torch_dtype=torch.bfloat16).cuda().eval()



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 859.62it/s]


## Embedding
Last real token of the final hidden state, L2-normalised. [B, 896]

In [3]:
@torch.no_grad()
def embed(texts):
    b = tok(texts, padding=True, truncation=True, max_length=256, return_tensors="pt").to("cuda")
    hidden = model(**b).last_hidden_state          # [B, T, 896]
    mask = b["attention_mask"]                     # [B, T]  1 = real token, 0 = padding
    idx = mask.sum(dim=1) - 1
    last = hidden[torch.arange(hidden.size(0), device=hidden.device), idx]   # [B, 896]                          
    return F.normalize( last.float(), dim=-1)

### Padding check
Same text alone vs in a padded batch. Should be ~1.0.

In [4]:
a = embed(["What port does SSH use?"])                               # alone, no padding
b = embed(["What port does SSH use?", "a much longer text " * 20])   # padded in a batch
print(F.cosine_similarity(a[0], b[0], dim=0))

tensor(0.9999, device='cuda:0')


## Data
First 20k pairs of NQ. Embedded once and cached to `nq20k_qwen05b.pt`.

In [5]:
ds = load_dataset("sentence-transformers/natural-questions", split="train").select(range(20000))

def embed_all(texts, bs=64):
    return torch.cat([embed(texts[i:i+bs]).cpu() for i in range(0, len(texts), bs)])

S = embed_all(ds["query"])    # [20000, 896]  states
A = embed_all(ds["answer"])   # [20000, 896]  actions
torch.save({"S": S, "A": A}, "nq20k_qwen05b.pt")

## Heads
Separate linear heads for states and actions, 896 → 512, unit length out.

In [6]:
class Head(nn.Module):
    def __init__(self, d_in=896, d_out=512):
        super().__init__()
        self.proj = nn.Linear(d_in, d_out, bias=True)

    def forward(self, x):                        # x: [B, 896]
        return F.normalize(self.proj(x), dim=-1) # [B, 512], unit length

state_head, action_head = Head().cuda(), Head().cuda()

### Shape check

In [7]:
data = torch.load("nq20k_qwen05b.pt")
s = state_head(data["S"][:8].cuda())
print(s.shape, s.norm(dim=-1))

torch.Size([8, 512]) tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)


## Loss
Symmetric InfoNCE, in-batch negatives, scale 20.

In [8]:
def info_nce(s, a, scale=20.0):             # s, a: [B, 512], unit length
    logits = scale * (s @ a.T)              # [B, B]
    t = torch.arange(s.size(0), device=s.device)
    return (F.cross_entropy(logits, t) + F.cross_entropy(logits.T, t)) / 2

## Train / test split
18k train, last 2k test. All on GPU.

In [9]:
S_train, A_train = S[:18000], A[:18000]
S_test, A_test = S[-2000:], A[-2000:]

In [10]:
print(next(state_head.parameters()).device, S_train.device)

cuda:0 cpu


In [11]:
S_train, A_train, S_test, A_test = [x.cuda() for x in (S_train, A_train, S_test, A_test)]

## Baseline: random batches
2,000 steps, batch 64. Top-1 retrieval on the test set, raw embeddings vs heads.

In [12]:
opt = torch.optim.AdamW(list(state_head.parameters()) + list(action_head.parameters()), lr=1e-3)

In [13]:
@torch.no_grad()
def top1(s, a):                                   # s, a: [N, d], unit length
    pred = (s @ a.T).argmax(dim=1)
    return (pred == torch.arange(len(s), device=s.device)).float().mean().item()

print("raw", top1(S_test, A_test))

for step in range(2001):
    idx = torch.randint(0, 18000, (64,))
    loss = info_nce(state_head(S_train[idx]), action_head(A_train[idx]))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 200 == 0:
        with torch.no_grad():
            print(step, round(loss.item(), 3),
                  top1(state_head(S_test), action_head(A_test)))

raw 0.005000000353902578
0 4.207 0.0
200 1.127 0.26600000262260437
400 0.436 0.32750001549720764
600 0.418 0.34800001978874207
800 0.243 0.34700000286102295
1000 0.327 0.38100001215934753
1200 0.416 0.38200002908706665
1400 0.307 0.3945000171661377
1600 0.232 0.39750000834465027
1800 0.155 0.42250001430511475
2000 0.194 0.4050000309944153


## Hard negatives

### Save B4 heads
Snapshot after the baseline run so both hard-negative runs start from the same place.

In [14]:
import copy
b4_state, b4_action = copy.deepcopy(state_head.state_dict()), copy.deepcopy(action_head.state_dict())

### Mine
For each train query, the highest-scoring wrong answer under the B4 heads.

In [15]:
with torch.no_grad():
    scores = state_head(S_train) @ action_head(A_train).T     # [18000, 18000]

In [16]:
scores.fill_diagonal_(float('-inf'))
hard_idx = scores.argmax(dim=1)                                # [18000]

### Loss with hard negatives
Each query scores B in-batch answers + B hard ones. [B, 2B]
Answer → query direction stays in-batch only.

In [17]:
def info_nce_hard(s, a, h, scale=20.0):        # s, a, h: [B, 512]
    logits = scale * (s @ torch.cat([a, h]).T)  # [B, 2B]
    t = torch.arange(s.size(0), device=s.device)
    return (F.cross_entropy(logits, t) + F.cross_entropy(logits[:, :s.size(0)].T, t)) / 2

### Two-stage vs from scratch
Run 1: B4 heads + 2,000 hard steps. Run 2: fresh heads, 4,000 hard steps.

In [18]:
def train(sh, ah, steps, hard):
    opt = torch.optim.AdamW(list(sh.parameters()) + list(ah.parameters()), lr=1e-3)
    for step in range(1, steps + 1):
        idx = torch.randint(0, 18000, (64,))
        s, a = sh(S_train[idx]), ah(A_train[idx])
        loss = info_nce_hard(s, a, ah(A_train[hard_idx[idx]])) if hard else info_nce(s, a)
        opt.zero_grad(); loss.backward(); opt.step()
        if step % 500 == 0:
            with torch.no_grad():
                print(step, round(loss.item(), 3), top1(sh(S_test), ah(A_test)))

del scores                                             # free the [18000, 18000] matrix

# Run 1, two-stage: B4 heads (2,000 random steps already done) → 2,000 hard steps
sh1, ah1 = Head().cuda(), Head().cuda()
sh1.load_state_dict(b4_state); ah1.load_state_dict(b4_action)
train(sh1, ah1, 2000, hard=True)

# Run 2, from-scratch: fresh heads → 4,000 hard steps
sh2, ah2 = Head().cuda(), Head().cuda()
train(sh2, ah2, 4000, hard=True)

500 0.411 0.3850000202655792
1000 0.362 0.4000000059604645
1500 0.341 0.41700002551078796
2000 0.237 0.4125000238418579
500 1.198 0.3215000033378601
1000 0.767 0.343500018119812
1500 0.475 0.3760000169277191
2000 0.305 0.38750001788139343
2500 0.343 0.40400001406669617
3000 0.314 0.41100001335144043
3500 0.237 0.40550002455711365
4000 0.249 0.42250001430511475


In [19]:
print([k for k, v in globals().items() if isinstance(v, list) and v and isinstance(v[0], str)])

['_ih', 'In']


In [20]:
Q, ANS = ds["query"][:18000], ds["answer"][:18000]

for i in [0, 1, 2, 3, 4]:
    print("Q:   ", Q[i])
    print("GOLD:", ANS[i][:150])
    print("HARD:", ANS[hard_idx[i]][:150], "\n")

Q:    when did richmond last play in a preliminary final
GOLD: Richmond Football Club Richmond began 2017 with 5 straight wins, a feat it had not achieved since 1995. A series of close losses hampered the Tigers t
HARD: List of Tampa Bay Lightning seasons Tampa Bay made their first playoffs in the 1995–96 season. The team's best year was the 2017–18 season, in which t 

Q:    who sang what in the world's come over you
GOLD: Jack Scott (singer) At the beginning of 1960, Scott again changed record labels, this time to Top Rank Records.[1] He then recorded four Billboard Hot
HARD: New Year's Day (U2 song) "New Year's Day" is a song by Irish rock band U2. It is on their 1983 album War and it was released as the album's lead singl 

Q:    who produces the most wool in the world
GOLD: Wool Global wool production is about 2 million tonnes per year, of which 60% goes into apparel. Wool comprises ca 3% of the global textile market, but
HARD: Cotton Current estimates for world production are abo

In [21]:
print([k for k, v in globals().items() if isinstance(v, list) and v and isinstance(v[0], str)])


['_ih', 'In', 'Q', 'ANS']


In [22]:
print({k: type(v).__name__ for k, v in globals().items() if "dataset" in type(v).__module__.lower() or k in ("ds", "data", "dataset")})

{'ds': 'Dataset', 'data': 'dict'}


In [23]:
print(ds)
print({k: (type(v).__name__, len(v)) for k, v in data.items()})

Dataset({
    features: ['query', 'answer'],
    num_rows: 20000
})
{'S': ('Tensor', 20000), 'A': ('Tensor', 20000)}


In [24]:
Q, ANS = ds["query"][:18000], ds["answer"][:18000]
for i in [0, 1, 2, 3, 4]:
    print("Q:   ", Q[i])
    print("GOLD:", ANS[i][:150])
    print("HARD:", ANS[hard_idx[i]][:150], "\n")

Q:    when did richmond last play in a preliminary final
GOLD: Richmond Football Club Richmond began 2017 with 5 straight wins, a feat it had not achieved since 1995. A series of close losses hampered the Tigers t
HARD: List of Tampa Bay Lightning seasons Tampa Bay made their first playoffs in the 1995–96 season. The team's best year was the 2017–18 season, in which t 

Q:    who sang what in the world's come over you
GOLD: Jack Scott (singer) At the beginning of 1960, Scott again changed record labels, this time to Top Rank Records.[1] He then recorded four Billboard Hot
HARD: New Year's Day (U2 song) "New Year's Day" is a song by Irish rock band U2. It is on their 1983 album War and it was released as the album's lead singl 

Q:    who produces the most wool in the world
GOLD: Wool Global wool production is about 2 million tonnes per year, of which 60% goes into apparel. Wool comprises ca 3% of the global textile market, but
HARD: Cotton Current estimates for world production are abo

In [25]:
def rank(state_text, action_texts):
    with torch.no_grad():
        s = state_head(embed([state_text]).cuda())      # [1, 512]
        a = action_head(embed(action_texts).cuda())     # [k, 512]
        scores = (s @ a.T).squeeze(dim=0)                     # [k]   cosine of the state with each action
    return scores

In [26]:
def choice(state_text, options):
    scores = rank(state_text, options)            # [k] cosines
    probs  = F.softmax(20.0 * scores, dim=0)       # [k], sums to 1
    pick   = options[probs.argmax()]
    conf   = probs.max().item()
    return pick, conf, probs

In [31]:
state = "who wrote the theory of relativity"
actions = ["Albert Einstein developed the theory of relativity, one of the two pillars of modern physics.",
           "Isaac Newton formulated the laws of motion and universal gravitation.",
           "Marie Curie conducted pioneering research on radioactivity.",
           "The Eiffel Tower is a wrought-iron lattice tower in Paris."]

In [32]:
pick, conf, probs = choice(
    state_text=state,
    options=actions)
print(pick[:40], round(conf, 3), probs.round(decimals=3).tolist())

Albert Einstein developed the theory of  0.821 [0.8209999799728394, 0.17800000309944153, 0.0020000000949949026, 0.0]


In [33]:
import time
def bench(fn, n=50):
    for _ in range(5): fn()                 # warm-up
    torch.cuda.synchronize(); t = time.perf_counter()
    for _ in range(n): fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t) / n * 1000   # ms per call

In [36]:
bench(lambda: rank(state, actions))                       # uncached, both calls

94.31165399999827

In [35]:
bench(lambda: state_head(embed([state]).cuda()))          # call 1 only ≈ the cached cost

51.45896999999877

In [37]:
for k in [1, 10, 100]:
    acts = (actions * 100)[:k]            # reuse your short action texts
    print(k, bench(lambda: action_head(embed(acts).cuda())))

1 51.78074999999808
10 43.940567999998166
100 43.936305999995966


In [ ]:
cache = {}
def rank_cached(state_text, action_texts):
    with torch.no_grad():
        s = state_head(embed([state_text]).cuda())      # [1, 512]
        if(action_texts not in cache):
            a = action_head(embed(action_texts).cuda()) 
        a = a
        scores = (s @ a.T).squeeze(dim=0)                     # [k]   cosine of the state with each action
    return scores